# Minutas verificables con agentes (ReAct + auditor) - Entrega 3

Ejecuta en Google Colab con `Entorno de ejecución -> Cambiar tipo de entorno de ejecución -> GPU (T4)`. Modelo: `ministral-3:3b-instruct-2512-q4_K_M` vía Ollama. Edita solo la celda 1 y luego `Ejecutar todo`; Colab pedirá subir archivos en las celdas 3 y 4.

**Modos (`MODE`):**
- `DEMO`: una sola reunión (M01) del split *test*, condición `full`. Es la corrida para el video: la celda 6 imprime la traza agente por agente.
- `VALIDATION`: split *validation*, condiciones `full` y `d2`, limitado a `LIMIT` reuniones (2 por defecto). Prueba de humo para ver si el modelo sigue el protocolo ReAct.
- `FULL`: split *test* completo, las 7 condiciones (`baseline, d2, full, no_react, no_auditor, no_fewshot, no_cot`). Es la evaluación final.

**Código:** con `SOURCE = 'zip'` se sube `deliverable3_colab.zip` (generado con `python scripts/package_for_colab.py`); con `SOURCE = 'git'` se clona `REPO_URL` y se fija `COMMIT`. Las respuestas del LLM se guardan en caché y los resultados por reunión se reanudan si la sesión se interrumpe.

In [ ]:
# 1. Configuración
MODE = 'DEMO'        # 'DEMO' | 'VALIDATION' | 'FULL'
LIMIT = 2            # solo VALIDATION: nº de reuniones (None = todas)
SOURCE = 'zip'       # 'zip' (subir deliverable3_colab.zip) | 'git'
REPO_URL = 'https://github.com/benjaminnalvear-dev/genai-vio-meeting-minutes'
COMMIT = ''          # solo SOURCE='git': hash o rama a hacer checkout ('' = HEAD)
MODEL = 'ministral-3:3b-instruct-2512-q4_K_M'

ALL_CONDITIONS = 'baseline,d2,full,no_react,no_auditor,no_fewshot,no_cot'
PLAN = {
    'DEMO':       dict(split='test',       conditions='full',    extra=['--demo']),
    'VALIDATION': dict(split='validation', conditions='full,d2', extra=(['--limit', str(LIMIT)] if LIMIT else [])),
    'FULL':       dict(split='test',       conditions=ALL_CONDITIONS, extra=[]),
}
assert MODE in PLAN, f'MODE invalido: {MODE}'
SPLIT, CONDITIONS, EXTRA = PLAN[MODE]['split'], PLAN[MODE]['conditions'], PLAN[MODE]['extra']
print('MODE =', MODE, '| split =', SPLIT, '| conditions =', CONDITIONS, '| extra =', EXTRA)

In [ ]:
# 2. GPU + Ollama + modelo (igual que la Entrega 2)
import subprocess, time, shutil, requests
if not shutil.which('nvidia-smi'):
    raise RuntimeError('Esta sesión de Colab no tiene GPU. Ve a Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU T4 y vuelve a ejecutar.')
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise RuntimeError('Colab no detecta una GPU NVIDIA activa. Selecciona GPU T4 y reconecta.')
print('GPU detectada:', gpu.stdout.strip())
if not shutil.which('zstd'):
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'zstd'], check=True)
if not shutil.which('ollama'):
    subprocess.run(['bash', '-o', 'pipefail', '-c', 'curl -fsSL https://ollama.com/install.sh | sh'], check=True)
if not shutil.which('ollama'):
    raise RuntimeError('La instalación terminó sin dejar el ejecutable ollama en PATH')

ollama_log = open('/content/ollama_serve.log', 'a')
server = subprocess.Popen(['ollama', 'serve'], stdout=ollama_log, stderr=subprocess.STDOUT)
for _ in range(90):
    try:
        if requests.get('http://127.0.0.1:11434/api/tags', timeout=2).ok:
            break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError('Ollama no inició; revisa /content/ollama_serve.log')

print('Descargando modelo; solo se hace una vez por sesión de Colab...')
subprocess.run(['ollama', 'pull', MODEL], check=True)
subprocess.run(['ollama', '--version'], check=False)
print('Modelo:', MODEL)

In [ ]:
# 3. Obtener el código y dependencias
import os, subprocess, zipfile
from google.colab import files

if SOURCE == 'zip':
    up = files.upload()  # sube deliverable3_colab.zip
    zname = next(n for n in up if n.endswith('.zip'))
    with zipfile.ZipFile(zname) as z:
        z.extractall('/content')
elif SOURCE == 'git':
    subprocess.run(['rm', '-rf', '/content/repo'], check=True)
    subprocess.run(['git', 'clone', REPO_URL, '/content/repo'], check=True)
    if COMMIT:
        subprocess.run(['git', '-C', '/content/repo', 'checkout', COMMIT], check=True)
    subprocess.run(['rm', '-rf', '/content/deliverable3'], check=True)
    subprocess.run(['cp', '-r', '/content/repo/deliverable3', '/content/deliverable3'], check=True)
else:
    raise ValueError("SOURCE debe ser 'zip' o 'git'")

!pip install -q -r /content/deliverable3/requirements.txt
%cd /content/deliverable3
print(open('COMMIT.txt').read() if os.path.exists('COMMIT.txt') else 'git ' + COMMIT)

In [ ]:
# 4. Subir el dataset (Fine Tuning.zip)
from pathlib import Path
up = files.upload()
DATA = str(Path(next(n for n in up if n.endswith('.zip'))).resolve())
print('DATA =', DATA)

In [ ]:
# 5. Ejecutar el experimento (salida en streaming; reanudable)
import subprocess, sys
RESULTS = f'results/{SPLIT}'
cmd = [sys.executable, '-u', 'scripts/run_experiment.py', '--data', DATA, '--split', SPLIT,
       '--conditions', CONDITIONS, '--out', RESULTS, '--traces', 'traces'] + EXTRA
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# 6. Traza agente por agente (condición `full`)
import json
from pathlib import Path

def short(x, n=300):
    s = x if isinstance(x, str) else json.dumps(x, ensure_ascii=False)
    s = ' '.join(s.split())
    return s if len(s) <= n else s[:n] + ' ...'

trace_files = sorted(Path('traces/full').glob('*.jsonl'))
MID = 'M01' if Path('traces/full/M01.jsonl').exists() else (trace_files[0].stem if trace_files else None)
print('Reunión:', MID)
for line in Path(f'traces/full/{MID}.jsonl').read_text(encoding='utf-8').splitlines():
    ev = json.loads(line)
    agent = ev.get('agent')
    if agent in ('router', 'finalize'):
        print(f"\n=== [{agent}] {short(ev.get('output') or ev.get('action') or ev, 400)}")
        continue
    print(f"\n[{agent} paso {ev.get('step')}] {ev.get('action')}   ({ev.get('seconds')}s, {ev.get('tokens')} tokens)")
    if ev.get('thought'):
        print('  Pensamiento :', short(ev['thought']))
    for tc in ev.get('tool_calls') or []:
        print('  Herramienta :', short(tc, 200))
    for ob in ev.get('observations') or []:
        print('  Observación :', short(ob, 200))
    if ev.get('output') is not None:
        print('  Salida      :', short(ev['output'], 200))

In [ ]:
# 7. Acta final y métricas
import json
import pandas as pd
from pathlib import Path
from IPython.display import display

acta = json.loads(Path(f'{RESULTS}/full/{MID}.json').read_text(encoding='utf-8'))
print(json.dumps(acta, ensure_ascii=False, indent=2)[:6000])

summary = json.loads(Path(f'{RESULTS}/summary.json').read_text(encoding='utf-8'))
conds = [c for c in summary if c not in ('meta', 'paired_full_vs_d2')]

def flat(d, pre=''):
    out = {}
    for k, v in d.items():
        if isinstance(v, dict):
            out.update(flat(v, f'{pre}{k}.'))
        elif isinstance(v, (int, float)) and not isinstance(v, bool):
            out[f'{pre}{k}'] = v
    return out

print('\nMétricas primarias')
display(pd.DataFrame({c: flat(summary[c]['primary']) for c in conds}).round(3))
print('Métricas extendidas')
display(pd.DataFrame({c: flat({k: v for k, v in summary[c]['extended'].items() if k != 'per_meeting'}) for c in conds}).round(3))

In [ ]:
# 8. Figuras
import subprocess, sys
from pathlib import Path
from IPython.display import Image, display
subprocess.run([sys.executable, 'scripts/make_figures.py', '--summary', f'{RESULTS}/summary.json', '--out', 'figures'], check=True)
for p in sorted(Path('figures').glob('*.png')):
    print(p)
    display(Image(str(p)))

In [ ]:
# 9. Empaquetar y descargar results/, traces/ y figures/
import subprocess
from pathlib import Path
from google.colab import files
folders = [d for d in ('results', 'traces', 'figures') if Path(d).exists()]
subprocess.run(['zip', '-qr', '/content/d3_outputs.zip'] + folders, check=True)
files.download('/content/d3_outputs.zip')